# 08 — Comprendre les fonctions (notebook didactique)

Ce notebook est fait pour **apprendre en lisant le code**. Il complète le notebook 07 (qui te
laisse *appeler* les fonctions) : ici, pour chaque fonction clé, tu trouves

1. **son vrai code source**, affiché en direct (`inspect.getsource`) — donc toujours à jour ;
2. un **décryptage** qui le déroule bloc par bloc en langage clair ;
3. un **exemple vivant** qui rend la figure ;
4. une **petite expérience** (« change tel paramètre, observe »).

Le parcours va du **simple au subtil**, et s'appuie sur de **vrais pièges** rencontrés sur le
projet — c'est en les comprenant qu'on progresse. À la fin, une **fiche des pièges pandas
récurrents**.

> À lancer depuis `notebooks/` (chemins `../data`). Les exemples utilisent le `donnees.csv` présent
> (fictif ou réel, peu importe pour comprendre le code).

In [1]:
import sys, inspect
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display, Code, Markdown
sys.path.insert(0, "../src")

from report_builder import (load_aphp, load_regional, load_survival, reconstruire_grains,
                            _market_share_evolution)
from chart_utils import (line_evolution, donut_market_share, survival_by_stage,
                         GHU_LIST, TREATMENT_COLS, REGIONAL_COLORS)

DATA_DIR = Path("../data").resolve()
aphp = load_aphp(DATA_DIR); reg = load_regional(DATA_DIR); surv = load_survival(DATA_DIR)

# Petits découpages réutilisés dans les exemples (cf. notebook 07).
def tot(df, e):
    return df[(df.entite == e) & (df.appareil == "TOTAL") & (df.organe == "TOTAL")].sort_values("annee")
def ghu_slice(df, appareil="TOTAL", organe="TOTAL"):
    return df[df.entite.isin(GHU_LIST) & (df.appareil == appareil) & (df.organe == organe)]

def source(fn):
    """Affiche le VRAI code source de la fonction (syntaxe colorée)."""
    return Code(inspect.getsource(fn), language="python")

def montre(fig):
    display(fig)

print("Prêt. Données chargées :", aphp.shape[0], "lignes AP-HP,",
      surv.shape[0], "lignes survie.")

Prêt. Données chargées : 2297 lignes AP-HP, 10084 lignes survie.


## A. `line_evolution` — le patron de base : filtrer → tracer

C'est **la** fonction à comprendre en premier : presque toutes les autres suivent le même schéma.
Elle prend un tableau déjà découpé, et trace une **courbe par entité** (une ligne par GHU, ou une
ligne par mode de séjour…). **Concept clé :** la fonction ne fait *pas* le découpage des données —
c'est l'appelant qui lui passe la bonne tranche. La fonction ne s'occupe que du **dessin**.

In [2]:
source(line_evolution)

def line_evolution(
    df: pd.DataFrame,
    x: str,
    y: str,
    group: str,
    title: str,
    y_label: str = "",
    entities: list = None,
    show_covid: bool = True,
    y_zero: bool = False,
) -> go.Figure:
    """Courbes d'évolution temporelle pour plusieurs entités."""
    fig = go.Figure()
    ents = entities or sorted(df[group].unique())

    for ent in ents:
        d = df[df[group] == ent].sort_values(x)
        vals = d[y].values
        pcts = [None] + [
            f"{(v - p) / p * 100:+.1f}%" if p else "N/A"
            for v, p in zip(vals[1:], vals[:-1])
        ]
        color = get_color(ent)
        width = 3 if ent == "AP-HP" else 2
        dash = "solid" if ent == "AP-HP" else "solid"

        fig.add_trace(go.Scatter(
            x=d[x], y=d[y],
            name=ent,
            mode="lines+markers",
            line=dict(color=color, width=width, dash=dash),
            marker=dict(size=8, color=color, line=dict(width=1.5, color="white")),
            customdata=pcts,
            hovertemplate=(
                f"<b>{ent}</b><br>"
                f"%{{y:,.0f}}<br>"
                f"Évol. N-1 : %{{customdata}}"
                "<extra></extra>"
            ),
        ))

    years_in_data = sorted(df[x].unique())
    if show_covid and 2020 in years_in_data:
        fig.add_vrect(
            x0=2019.5, x1=2020.5,
            fillcolor="#FFE8E8", opacity=0.5,
            line_width=0, layer="below",
            annotation_text="COVID-19",
            annotation_position="top left",
            annotation_font_size=11,
            annotation_font_color="#E63946",
        )

    lo = _layout()
    if y_zero:
        lo["yaxis"] = dict(showgrid=True, gridcolor="#F0F0F0", zeroline=False, rangemode="tozero")
    # Axe X = années entières : un tic par an, format entier (sinon Plotly insère des
    # demi-années « 2022,5 » sur un axe numérique). Nouveau dict (BASE_LAYOUT.copy()
    # est superficiel → ne pas muter xaxis en place).
    lo["xaxis"] = {**lo["xaxis"], "tickmode": "linear", "dtick": 1, "tickformat": "d"}
    if years_in_data:
        lo["xaxis"]["tick0"] = int(years_in_data[0])
    fig.update_layout(
        title=dict(text=title, font_size=17),
        xaxis_title="Année",
        yaxis_title=y_label,
        **lo,
    )
    return fig

### Décryptage

- **Signature** : `line_evolution(df, x, y, group, title, ...)`. On lui dit quelle colonne est
  l'axe X (`annee`), laquelle est l'axe Y (`nb_patients`), et par quoi grouper les courbes
  (`group="entite"`). C'est **générique** : la même fonction trace patients, séjours, parts de marché.
- **Une trace par groupe** : la fonction boucle sur les valeurs distinctes de `group` et ajoute une
  courbe Plotly (`go.Scatter`) par groupe. C'est le motif Plotly de base.
- **`entities`** (optionnel) : permet de **fixer l'ordre et la sélection** des courbes (ex. les 6
  GHU dans un ordre voulu). Si on ne le passe pas, la fonction prend les valeurs présentes.
- **`show_covid`, `y_zero`** : des options de présentation (repère COVID, axe Y qui démarre à 0).
- **Ce qu'elle NE fait pas** : aucun filtre métier. Si tu lui passes un tableau contenant plusieurs
  appareils, elle mélangera tout. D'où l'importance du **découpage en amont** (`tot(...)`).

In [3]:
# Exemple : évolution du nombre de patients AP-HP (une seule entité → une courbe)
montre(line_evolution(tot(aphp, "AP-HP"), "annee", "nb_patients", "entite",
                      "Évolution des patients — AP-HP"))

### À toi de jouer

Change `"nb_patients"` en `"nb_nouveaux_patients"`, ou remplace `tot(aphp, "AP-HP")` par
`ghu_slice(aphp)` avec `group="entite"` et `entities=GHU_LIST` → tu obtiens une courbe par GHU.
Observe : **la fonction n'a pas changé**, seul le **découpage** en entrée change le résultat.

## B. `donut_market_share` — le piège du paramètre par défaut

Un donut de répartition (parts de marché). **On l'a vue tomber en panne** : le donut « répartition
par type d'établissement » sortait vide. La cause est une **ligne discrète** — un excellent cas pour
apprendre à lire un paramètre par défaut.

In [ ]:
source(donut_market_share)

### Décryptage — et le piège

Repère la ligne :

```python
ents = entities or GHU_LIST
```

C'est un idiome Python : « **si `entities` est fourni, l'utiliser ; sinon, prendre `GHU_LIST`** ».
Pratique… mais **dangereux** ici. La fonction ne garde ensuite que les lignes dont l'entité est dans
`ents`. Donc :

- Si tu l'appelles **sans** `entities` sur un tableau **régional** (dont les entités sont
  `AP-HP`, `CH`, `CHU`… — **aucun GHU**), alors `ents = GHU_LIST` → **aucune** ligne ne matche →
  **donut vide**.
- Le vrai code des pages passe toujours `entities=types_etab` → il marche. C'est un exemple
  d'appel qui l'oubliait qui échouait.

**Leçon générale :** un paramètre par défaut « pratique » (`x or DÉFAUT`) crée un **couplage
implicite**. Quand tu réutilises une fonction dans un contexte différent, vérifie ses défauts —
ils supposent souvent un contexte précis (ici : « les entités sont des GHU »).

In [ ]:
# CORRECT : on précise les entités présentes dans le tableau régional
reg_last = reg[(reg.appareil == "TOTAL") & (reg.organe == "TOTAL")]
reg_last = reg_last[reg_last.annee == reg_last.annee.max()]
montre(donut_market_share(reg_last, "entite", "nb_patients",
                          "Répartition par type d'établissement",
                          entities=sorted(reg_last.entite.unique())))

### À toi de jouer

Retire l'argument `entities=...` de l'appel ci-dessus et ré-exécute : le donut redevient **vide**
(les entités régionales ne sont pas des GHU). C'est *exactement* le bug qu'on a corrigé — reproduis-le
pour bien le sentir.

## C. `survival_by_stage` — deux subtilités métier dans une fonction

Instantané de la survie **par stade** (I-III vs IV) pour une année. Elle concentre **deux leçons** :
un filtre métier indispensable, et une gestion d'année qu'on a récemment corrigée.

In [ ]:
source(survival_by_stage)

### Décryptage

**1. Filtrer la population (sinon double-comptage).** Le schéma long contient la survie en **double**
(populations `tous` et `nouveaux`). La fonction filtre donc **impérativement** une `population` :

```python
& (df_surv["population"] == population)
```

Si on l'oubliait, chaque stade apparaîtrait deux fois. C'est un **invariant du format long** : dès
qu'une dimension (`population`, `stade`, `age`…) existe en plusieurs valeurs, il faut en **choisir
une** avant d'agréger ou d'afficher. Retiens ce réflexe : *filtrer avant d'agréger*.

**2. Le repli sur la dernière année disponible.** Regarde :

```python
if year is None and not d.empty:
    year = int(d["annee"].max())
d = d[d["annee"] == year]
```

Si l'appelant ne passe **pas** d'année, la fonction prend la **dernière année qui a de la survie**.
Mais si on lui force `year = derniere_annee_du_dataset` (ex. 2025), et que 2025 n'a **pas encore** de
survie à 5 ans (il faut 5 ans de recul !), alors `d` est vide → le graphe affiche « Pas de données ».
**C'est le bug qu'on a corrigé** : dans les pages, on n'a plus qu'à **ne pas forcer l'année**, et la
fonction retombe sur la dernière année *réellement disponible*.

**Leçon :** une valeur par défaut « intelligente » (`year is None → dernière dispo`) ne protège que
si on la **laisse agir**. La court-circuiter avec une valeur qui semble raisonnable (« l'année la
plus récente ») mais qui ne convient pas au *contenu* (la survie récente n'existe pas) casse le repli.

In [ ]:
# On NE force PAS l'année → la fonction prend la dernière année AVEC survie
montre(survival_by_stage(surv, "AP-HP", "SEIN"))

### À toi de jouer

Passe explicitement une année sans survie, ex. `year=2019` (ou une année absente) : tu verras le
message « Pas de données de survie ». Puis enlève l'argument → le graphe se remplit. Compare les deux :
c'est la différence entre *forcer* une année et *laisser* la fonction choisir.

## D. `reconstruire_grains` — fabriquer les niveaux agrégés manquants

Le format long stocke le grain **le plus fin** (par organe). Les grains agrégés (par appareil, et
global `TOTAL/TOTAL`) sont **reconstruits à la lecture** par les `load_*`. Cette fonction unifie
cette reconstruction (elle nous a occupés : plusieurs graphes régionaux/délais étaient vides parce
qu'un `load_*` ne l'appelait pas).

In [ ]:
source(reconstruire_grains)

### Décryptage

- **Deux étages** : `_add_organe_total` fabrique, pour chaque (entité, année, appareil), la ligne
  `organe = TOTAL` en agrégeant ses organes ; puis `_add_appareil_total` fabrique la ligne globale
  `appareil = TOTAL / organe = TOTAL` en agrégeant les appareils.
- **Somme vs moyenne** : les **comptes** (patients, séjours) s'**additionnent** ; les **délais**
  (médianes) ne s'additionnent pas → on en prend la **moyenne**. La fonction sait traiter les deux
  (paramètres `comptes` / `delais`).
- **La garde « seulement si absent »** : la reconstruction ne s'applique **que** si le grain TOTAL
  n'existe pas déjà. En réel, les sources fournissent parfois ces lignes (ex. `AP-HP Total`) :
  on ne les **écrase pas** (crucial pour ne pas re-sommer et casser le dédoublonnage AP-HP).
  En fictif, elles sont absentes → on les reconstruit.

**Leçon d'architecture :** stocker le grain canonique le plus fin et **reconstruire** les agrégats
à la lecture évite d'avoir à maintenir les totaux cohérents dans les données. Mais tout consommateur
qui a besoin d'un grain agrégé doit passer par la reconstruction — sinon vide (le bug qu'on a eu).

In [ ]:
# Démonstration : partir du grain organe seul, reconstruire les TOTAL
petit = reg[(reg.entite == "AP-HP") & (reg.appareil != "TOTAL") & (reg.organe != "TOTAL")].copy()
print("avant — grains présents :",
      petit.assign(g=petit.appareil.eq("TOTAL").astype(str)+"/"+petit.organe.eq("TOTAL").astype(str))
           .g.value_counts().to_dict())
recon = reconstruire_grains(petit)
print("après — lignes TOTAL/TOTAL :",
      len(recon[(recon.appareil == "TOTAL") & (recon.organe == "TOTAL")]))

### À toi de jouer

Regarde ce que devient une valeur agrégée : compare `recon` filtré sur `appareil=="TOTAL"` à la
somme manuelle des organes d'un appareil pour une année. Tu retrouveras la somme (comptes) — et pour
les délais, la moyenne, pas la somme.

## E. Le piège d'alignement d'index pandas (le bug le plus vicieux)

Pas une fonction de graphe, mais **le** piège pandas à comprendre — il nous a coûté un vrai bug
(l'AP-HP régional entièrement faux). Il est **invisible** tant que les données sont « bien rangées »,
et catastrophique sinon. Le comprendre te rendra bien plus autonome.

**L'idée :** quand tu assignes une `Series` à une colonne d'un `DataFrame`, pandas **aligne sur
l'index**, pas sur la position. Si les deux n'ont pas le même index, les valeurs se retrouvent sur
les **mauvaises lignes** (ou deviennent `NaN`).

In [ ]:
# Reproduction minimale du bug
df = pd.DataFrame({"niveau": ["a", "b", "c", "d"], "nb": [10, 20, 30, 40]})
keep = pd.Series([True, False, True, True])
df2 = df[keep].copy()                    # on filtre → index [0, 2, 3] : NON contigu !
print("index de df2 (filtré) :", list(df2.index))

# CASSÉ : on construit le cadre avec .values (numpy) → il prend un RangeIndex 0,1,2
cadre_bad = pd.DataFrame({"annee": df2["niveau"].values})     # index 0,1,2
cadre_bad["nb"] = df2["nb"].map(float)                        # Series index [0,2,3]
print("CASSÉ   (.values) :", cadre_bad["nb"].tolist(), " ← valeurs perdues / décalées")

# CORRIGÉ : on construit le cadre avec la Series → il hérite de l'index [0,2,3]
cadre_ok = pd.DataFrame({"annee": df2["niveau"]})             # index [0,2,3]
cadre_ok["nb"] = df2["nb"].map(float)                         # Series index [0,2,3] → aligné
print("CORRIGÉ (Series)  :", cadre_ok["nb"].tolist(), " ← tout est là, au bon endroit")

### Décryptage — pourquoi c'était invisible

- Après un **filtre** (`df[keep]`), l'index devient **non contigu** (`[0, 2, 3]`). Les valeurs
  numpy (`.values`) n'ont **pas d'index** : mises dans un `DataFrame`, elles créent un `RangeIndex`
  `0,1,2`. Une `Series` assignée ensuite (index `[0,2,3]`) s'**aligne** dessus → décalage/`NaN`.
- **Pourquoi ça a échappé aux tests** : le test synthétique d'origine avait les lignes utiles **en
  tête de fichier** (index contigu `0,1,2` = les positions) → l'alignement « tombait juste » par
  hasard. Sur les vrais fichiers, les lignes AP-HP étaient **après** des blocs Hôpital/GHU → index
  décalé → bug. La correction du test : une **fixture à index non contigu** (lignes entrelacées).
- **Le correctif réel** a été d'une ligne : passer `annee` en **`Series`** (pas `.values`) pour que
  le cadre hérite du bon index, exactement comme dans la reproduction ci-dessus.

**Leçons pandas (à graver) :**
1. `.values` / `.to_numpy()` **jette l'index** → assignation par **position**.
2. Une `Series` → assignation par **index** (alignement).
3. **Ne mélange pas** les deux sans y penser. Après un filtre, soit tu fais
   `df = df[keep].reset_index(drop=True)` (index propre 0..N), soit tu restes cohérent en `Series`.

## F. Fiche — pièges pandas récurrents sur ce projet

Une page à relire avant de coder un nouveau `load_*` ou une nouvelle figure.

| Piège | Symptôme | Réflexe |
|---|---|---|
| **Alignement d'index** | valeurs décalées / `NaN` après un filtre | `reset_index(drop=True)` après filtre, ou rester en `Series` (pas `.values`) |
| **Filtrer avant d'agréger** | chaque stade/entité compté 2× | choisir **une** `population` / `stade` avant somme ou affichage |
| **Défaut implicite** (`x or DÉFAUT`) | figure vide dans un autre contexte | vérifier les défauts d'une fonction avant de la réutiliser (ex. `entities or GHU_LIST`) |
| **Grain manquant** | graphe TOTAL/appareil vide | passer par `reconstruire_grains` ; ne reconstruire que si **absent** |
| **Année forcée** | « Pas de données » sur l'année récente | laisser le repli agir (survie : dernière année *disponible*, pas dernière année du dataset) |
| **Libellé qui ne matche pas** | section vide **silencieuse** en réel | inspecter les modalités observées (notebook 06) ; mojibake / espaces / casse |
| **Somme vs moyenne** | totaux de délais aberrants | comptes = **somme** ; médianes/taux = **moyenne** (pondérée si besoin) |

**Le fil rouge :** la plupart de ces bugs sont **invisibles en fictif** (données bien rangées,
tous les grains/années présents) et ne sortent que sur **données réelles**. D'où la valeur d'un jeu
de tests exerçant les cas réels (index non contigu, année sans survie, libellé exotique).

---

*Pour aller plus loin : le notebook 07 (`explorer les graphiques`) pour appeler les fonctions, le
guide `docs/guide_descriptif_sources.md` pour le format des sources, et `contrat_donnees_pivot.md`
pour le format long produit.*